# Advanced Parking Lot LLD

This notebook contains the final end-to-end Parking Lot LLD implementation discussed.

## Covered
- Vehicle, ParkingSpot, ParkingFloor, ParkingLot
- EntryGate and ExitGate
- ParkingTicket and Receipt
- ParkingPolicy
- CompositeSpotAssignmentStrategy
- FuelType + charger support for electric vehicles
- PricingStrategy + RateRepository
- PaymentStrategy + PaymentStrategyFactory using registry/decorator
- PaymentResult
- Duplicate vehicle entry check
- End-to-end tests


## Final Implementation

In [ ]:

from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import datetime, timedelta
from enum import Enum
from typing import Optional
import math
import uuid


# ======================================================
# ENUMS
# ======================================================

class VehicleType(Enum):
    MOTORCYCLE = "MOTORCYCLE"
    CAR = "CAR"
    TRUCK = "TRUCK"


class VehicleSize(Enum):
    SMALL = 1
    MEDIUM = 2
    LARGE = 3


class FuelType(Enum):
    PETROL = "PETROL"
    DIESEL = "DIESEL"
    ELECTRIC = "ELECTRIC"


class SpotType(Enum):
    BIKE_SPOT = "BIKE_SPOT"
    COMPACT_SPOT = "COMPACT_SPOT"
    LARGE_SPOT = "LARGE_SPOT"


class TicketStatus(Enum):
    ACTIVE = "ACTIVE"
    PAID = "PAID"
    LOST = "LOST"
    CANCELLED = "CANCELLED"


class PaymentStatus(Enum):
    SUCCESS = "SUCCESS"
    FAILED = "FAILED"
    PENDING = "PENDING"


# ======================================================
# VEHICLE
# ======================================================

class Vehicle:
    def __init__(
        self,
        license_plate: str,
        vehicle_size: VehicleSize,
        vehicle_type: VehicleType,
        fuel_type: FuelType,
    ):
        self.license_plate = license_plate
        self.vehicle_size = vehicle_size
        self.vehicle_type = vehicle_type
        self.fuel_type = fuel_type


class Car(Vehicle):
    def __init__(self, license_plate: str, fuel_type: FuelType = FuelType.PETROL):
        super().__init__(
            license_plate=license_plate,
            vehicle_size=VehicleSize.MEDIUM,
            vehicle_type=VehicleType.CAR,
            fuel_type=fuel_type,
        )


class Motorcycle(Vehicle):
    def __init__(self, license_plate: str, fuel_type: FuelType = FuelType.PETROL):
        super().__init__(
            license_plate=license_plate,
            vehicle_size=VehicleSize.SMALL,
            vehicle_type=VehicleType.MOTORCYCLE,
            fuel_type=fuel_type,
        )


class Truck(Vehicle):
    def __init__(self, license_plate: str, fuel_type: FuelType = FuelType.DIESEL):
        super().__init__(
            license_plate=license_plate,
            vehicle_size=VehicleSize.LARGE,
            vehicle_type=VehicleType.TRUCK,
            fuel_type=fuel_type,
        )


# ======================================================
# PARKING SPOT
# ======================================================

class ParkingSpot:
    def __init__(
        self,
        spot_id: str,
        spot_size: VehicleSize,
        spot_type: SpotType,
        has_charger: bool = False,
    ):
        self.spot_id = spot_id
        self.spot_size = spot_size
        self.spot_type = spot_type
        self.has_charger = has_charger
        self.is_occupied = False
        self.parked_vehicle = None

    def is_available(self) -> bool:
        return not self.is_occupied

    def park_vehicle(self, vehicle: Vehicle):
        if not self.is_available():
            raise Exception(f"Parking spot {self.spot_id} is already occupied")

        self.parked_vehicle = vehicle
        self.is_occupied = True

    def remove_vehicle(self):
        if self.is_available():
            raise Exception(f"Parking spot {self.spot_id} is already empty")

        removed_vehicle = self.parked_vehicle
        self.parked_vehicle = None
        self.is_occupied = False
        return removed_vehicle


# ======================================================
# PARKING POLICY
# ======================================================

class ParkingPolicy:
    """
    Responsible for eligibility:
    Can this vehicle park in this spot?
    """

    def __init__(self):
        self.allowed_spot_mapping = {
            VehicleType.MOTORCYCLE: [SpotType.BIKE_SPOT],
            VehicleType.CAR: [SpotType.COMPACT_SPOT, SpotType.LARGE_SPOT],
            VehicleType.TRUCK: [SpotType.LARGE_SPOT],
        }

    def can_park_vehicle_in_spot(self, vehicle: Vehicle, spot: ParkingSpot) -> bool:
        if not spot.is_available():
            return False

        if vehicle.vehicle_type not in self.allowed_spot_mapping:
            return False

        if spot.spot_type not in self.allowed_spot_mapping[vehicle.vehicle_type]:
            return False

        if spot.spot_size.value < vehicle.vehicle_size.value:
            return False

        return True


# ======================================================
# SPOT ASSIGNMENT STRATEGY
# ======================================================

class SpotAssignStrategy(ABC):
    @abstractmethod
    def assign_spot(self, floors: list, vehicle: Vehicle) -> Optional[ParkingSpot]:
        pass


class CompositeSpotAssignmentStrategy(SpotAssignStrategy):
    """
    Composite rules:
    1. Filter compatible spots using ParkingPolicy
    2. If vehicle is electric, prefer charging spot
    3. Choose smallest suitable spot, preserving large spots
    """

    def __init__(self, parking_policy: ParkingPolicy):
        self.parking_policy = parking_policy

    def assign_spot(self, floors: list, vehicle: Vehicle) -> Optional[ParkingSpot]:
        compatible_spots = []

        for floor in floors:
            for spot in floor.parking_spots:
                if self.parking_policy.can_park_vehicle_in_spot(vehicle, spot):
                    compatible_spots.append(spot)

        if not compatible_spots:
            return None

        if vehicle.fuel_type == FuelType.ELECTRIC:
            charging_spots = [
                spot for spot in compatible_spots
                if spot.has_charger
            ]

            if charging_spots:
                return min(charging_spots, key=lambda spot: spot.spot_size.value)

        return min(compatible_spots, key=lambda spot: spot.spot_size.value)


# ======================================================
# PARKING FLOOR
# ======================================================

class ParkingFloor:
    def __init__(self, floor_number: int):
        self.floor_number = floor_number
        self.parking_spots = []

    def add_parking_spot(self, parking_spot: ParkingSpot):
        for spot in self.parking_spots:
            if spot.spot_id == parking_spot.spot_id:
                raise Exception(f"Spot with id {parking_spot.spot_id} already exists")

        self.parking_spots.append(parking_spot)

    def remove_parking_spot(self, spot_id: str) -> bool:
        for spot in self.parking_spots:
            if spot.spot_id == spot_id:
                if not spot.is_available():
                    raise Exception("Cannot remove occupied parking spot")

                self.parking_spots.remove(spot)
                return True

        return False

    def get_all_available_spots_count(self) -> int:
        return sum(1 for spot in self.parking_spots if spot.is_available())


# ======================================================
# PARKING TICKET
# ======================================================

class ParkingTicket:
    def __init__(self, ticket_id: str, vehicle: Vehicle, parking_spot: ParkingSpot):
        self.ticket_id = ticket_id
        self.vehicle = vehicle
        self.parking_spot = parking_spot
        self.entry_time = datetime.now()
        self.exit_time = None
        self.status = TicketStatus.ACTIVE

    def is_active(self) -> bool:
        return self.status == TicketStatus.ACTIVE

    def mark_paid(self):
        if not self.is_active():
            raise Exception("Only active ticket can be marked as paid")

        if self.exit_time is None:
            self.exit_time = datetime.now()

        self.status = TicketStatus.PAID

    def mark_lost(self):
        if not self.is_active():
            raise Exception("Only active ticket can be marked as lost")
        self.status = TicketStatus.LOST

    def mark_cancelled(self):
        if not self.is_active():
            raise Exception("Only active ticket can be cancelled")
        self.status = TicketStatus.CANCELLED


# ======================================================
# RATE REPOSITORY
# ======================================================

class RateRepository(ABC):
    @abstractmethod
    def get_rate(self, vehicle_type: VehicleType, fuel_type: FuelType) -> float:
        pass


class InMemoryRateRepository(RateRepository):
    def __init__(self):
        self.rates = {
            (VehicleType.MOTORCYCLE, FuelType.PETROL): 20,
            (VehicleType.MOTORCYCLE, FuelType.ELECTRIC): 25,

            (VehicleType.CAR, FuelType.PETROL): 50,
            (VehicleType.CAR, FuelType.DIESEL): 50,
            (VehicleType.CAR, FuelType.ELECTRIC): 60,

            (VehicleType.TRUCK, FuelType.DIESEL): 100,
            (VehicleType.TRUCK, FuelType.ELECTRIC): 120,
        }

    def get_rate(self, vehicle_type: VehicleType, fuel_type: FuelType) -> float:
        key = (vehicle_type, fuel_type)

        if key not in self.rates:
            raise Exception(
                f"No rate configured for vehicle_type={vehicle_type}, fuel_type={fuel_type}"
            )

        return self.rates[key]


# ======================================================
# PRICING STRATEGY + RESOLVER
# ======================================================

class PricingStrategy(ABC):
    @abstractmethod
    def calculate_price(self, ticket: ParkingTicket) -> float:
        pass


class HourlyPricingStrategy(PricingStrategy):
    def __init__(self, rate_repository: RateRepository):
        self.rate_repository = rate_repository

    def calculate_price(self, ticket: ParkingTicket) -> float:
        if ticket.exit_time is None:
            raise Exception("Exit time is required to calculate price")

        duration = ticket.exit_time - ticket.entry_time
        total_seconds = duration.total_seconds()

        if total_seconds < 0:
            raise Exception("Exit time cannot be before entry time")

        hours = max(1, math.ceil(total_seconds / 3600))

        rate = self.rate_repository.get_rate(
            ticket.vehicle.vehicle_type,
            ticket.vehicle.fuel_type,
        )

        return hours * rate


@dataclass
class PricingContext:
    is_lost_ticket: bool = False
    is_weekend: bool = False
    is_event_day: bool = False
    has_membership_discount: bool = False


class PricingStrategyResolver:
    def __init__(self):
        self.default_strategy = None
        self.lost_ticket_strategy = None
        self.weekend_strategy = None
        self.event_day_strategy = None

    def set_default_strategy(self, strategy: PricingStrategy):
        self.default_strategy = strategy

    def set_lost_ticket_strategy(self, strategy: PricingStrategy):
        self.lost_ticket_strategy = strategy

    def set_weekend_strategy(self, strategy: PricingStrategy):
        self.weekend_strategy = strategy

    def set_event_day_strategy(self, strategy: PricingStrategy):
        self.event_day_strategy = strategy

    def resolve_strategy(self, pricing_context: PricingContext) -> PricingStrategy:
        if pricing_context.is_lost_ticket:
            if self.lost_ticket_strategy is None:
                raise Exception("Lost ticket pricing strategy is not configured")
            return self.lost_ticket_strategy

        if pricing_context.is_event_day:
            if self.event_day_strategy is None:
                raise Exception("Event day pricing strategy is not configured")
            return self.event_day_strategy

        if pricing_context.is_weekend:
            if self.weekend_strategy is None:
                raise Exception("Weekend pricing strategy is not configured")
            return self.weekend_strategy

        if self.default_strategy is None:
            raise Exception("Default pricing strategy is not configured")

        return self.default_strategy


# ======================================================
# PAYMENT STRATEGY + FACTORY
# ======================================================

@dataclass
class PaymentResult:
    transaction_id: str
    amount: float
    payment_method: str
    status: PaymentStatus
    message: str
    paid_at: datetime


class PaymentStrategy(ABC):
    @abstractmethod
    def process_payment(self, ticket_id: str, amount: float) -> PaymentResult:
        pass


class PaymentStrategyFactory:
    _registry = {}

    @classmethod
    def register_payment_strategy(cls, payment_method: str):
        def decorator(strategy_class):
            cls._registry[payment_method.upper()] = strategy_class
            return strategy_class
        return decorator

    @classmethod
    def get_payment_strategy(cls, payment_method: str) -> PaymentStrategy:
        method = payment_method.upper()

        if method not in cls._registry:
            raise Exception(f"No payment strategy registered for payment method: {payment_method}")

        strategy_class = cls._registry[method]
        return strategy_class()


@PaymentStrategyFactory.register_payment_strategy("UPI")
class UpiPaymentStrategy(PaymentStrategy):
    def process_payment(self, ticket_id: str, amount: float) -> PaymentResult:
        return PaymentResult(
            transaction_id=str(uuid.uuid4()),
            amount=amount,
            payment_method="UPI",
            status=PaymentStatus.SUCCESS,
            message=f"UPI payment successful for ticket {ticket_id}",
            paid_at=datetime.now(),
        )


@PaymentStrategyFactory.register_payment_strategy("CARD")
class CardPaymentStrategy(PaymentStrategy):
    def process_payment(self, ticket_id: str, amount: float) -> PaymentResult:
        return PaymentResult(
            transaction_id=str(uuid.uuid4()),
            amount=amount,
            payment_method="CARD",
            status=PaymentStatus.SUCCESS,
            message=f"Card payment successful for ticket {ticket_id}",
            paid_at=datetime.now(),
        )


@PaymentStrategyFactory.register_payment_strategy("CASH")
class CashPaymentStrategy(PaymentStrategy):
    def process_payment(self, ticket_id: str, amount: float) -> PaymentResult:
        return PaymentResult(
            transaction_id=str(uuid.uuid4()),
            amount=amount,
            payment_method="CASH",
            status=PaymentStatus.SUCCESS,
            message=f"Cash payment successful for ticket {ticket_id}",
            paid_at=datetime.now(),
        )


# ======================================================
# RECEIPT
# ======================================================

class Receipt:
    def __init__(
        self,
        receipt_id: str,
        ticket_id: str,
        vehicle_number: str,
        amount_paid: float,
        payment_method: str,
        transaction_id: str,
        payment_status: PaymentStatus,
        entry_time: datetime,
        exit_time: datetime,
        paid_at: datetime,
        parking_spot_id: str,
        status: str = "SUCCESS",
    ):
        self.receipt_id = receipt_id
        self.ticket_id = ticket_id
        self.vehicle_number = vehicle_number
        self.amount_paid = amount_paid
        self.amount = amount_paid  # alias for testing/printing convenience
        self.payment_method = payment_method
        self.transaction_id = transaction_id
        self.payment_status = payment_status
        self.entry_time = entry_time
        self.exit_time = exit_time
        self.paid_at = paid_at
        self.parking_spot_id = parking_spot_id
        self.status = status

    def __str__(self):
        return (
            f"Receipt ID: {self.receipt_id}\n"
            f"Ticket ID: {self.ticket_id}\n"
            f"Vehicle Number: {self.vehicle_number}\n"
            f"Spot ID: {self.parking_spot_id}\n"
            f"Amount Paid: {self.amount_paid}\n"
            f"Payment Method: {self.payment_method}\n"
            f"Transaction ID: {self.transaction_id}\n"
            f"Payment Status: {self.payment_status}\n"
            f"Entry Time: {self.entry_time}\n"
            f"Exit Time: {self.exit_time}\n"
            f"Paid At: {self.paid_at}\n"
            f"Receipt Status: {self.status}"
        )


# ======================================================
# PARKING LOT
# ======================================================

class ParkingLot:
    def __init__(
        self,
        parking_lot_id: str,
        name: str,
        address: str,
        pricing_strategy_resolver: PricingStrategyResolver,
        payment_strategy_factory: PaymentStrategyFactory,
        spot_assignment_strategy: SpotAssignStrategy,
    ):
        self.parking_lot_id = parking_lot_id
        self.name = name
        self.address = address
        self.parking_floors = []
        self.active_tickets = {}
        self.ticket_counter = 0

        self.pricing_strategy_resolver = pricing_strategy_resolver
        self.payment_strategy_factory = payment_strategy_factory
        self.spot_assignment_strategy = spot_assignment_strategy

    def add_parking_floor(self, parking_floor: ParkingFloor):
        for floor in self.parking_floors:
            if floor.floor_number == parking_floor.floor_number:
                raise Exception(f"Floor {parking_floor.floor_number} already exists")

        self.parking_floors.append(parking_floor)

    def remove_parking_floor(self, floor_number: int) -> bool:
        for floor in self.parking_floors:
            if floor.floor_number == floor_number:
                if floor.get_all_available_spots_count() != len(floor.parking_spots):
                    raise Exception("Cannot remove floor because some spots are occupied")

                self.parking_floors.remove(floor)
                return True

        return False

    def is_vehicle_already_parked(self, license_plate: str) -> bool:
        return any(
            ticket.vehicle.license_plate == license_plate
            for ticket in self.active_tickets.values()
            if ticket.is_active()
        )

    def park_vehicle(self, vehicle: Vehicle) -> ParkingTicket:
        if self.is_vehicle_already_parked(vehicle.license_plate):
            raise Exception(f"Vehicle {vehicle.license_plate} is already parked")

        spot = self.spot_assignment_strategy.assign_spot(
            self.parking_floors,
            vehicle,
        )

        if spot is None:
            raise Exception(f"No available parking spot for vehicle {vehicle.license_plate}")

        spot.park_vehicle(vehicle)

        self.ticket_counter += 1
        ticket_id = f"{self.parking_lot_id}-{self.ticket_counter}"

        ticket = ParkingTicket(ticket_id, vehicle, spot)
        self.active_tickets[ticket_id] = ticket

        return ticket

    def exit_vehicle(
        self,
        ticket_id: str,
        payment_method: str,
        pricing_context: PricingContext,
    ) -> Receipt:
        if ticket_id not in self.active_tickets:
            raise Exception(f"Invalid ticket id {ticket_id}")

        ticket = self.active_tickets[ticket_id]

        if not ticket.is_active():
            raise Exception(f"Ticket with id {ticket_id} is not active")

        # Resolve payment strategy before mutating ticket exit_time.
        payment_strategy = self.payment_strategy_factory.get_payment_strategy(payment_method)

        ticket.exit_time = datetime.now()

        price_strategy = self.pricing_strategy_resolver.resolve_strategy(pricing_context)
        price = price_strategy.calculate_price(ticket)

        payment_result = payment_strategy.process_payment(ticket.ticket_id, price)

        if payment_result.status != PaymentStatus.SUCCESS:
            ticket.exit_time = None
            raise Exception(f"Payment failed for ticket {ticket_id}: {payment_result.message}")

        ticket.mark_paid()

        receipt = Receipt(
            receipt_id=f"R-{ticket_id}",
            ticket_id=ticket.ticket_id,
            vehicle_number=ticket.vehicle.license_plate,
            amount_paid=price,
            payment_method=payment_result.payment_method,
            transaction_id=payment_result.transaction_id,
            payment_status=payment_result.status,
            entry_time=ticket.entry_time,
            exit_time=ticket.exit_time,
            paid_at=payment_result.paid_at,
            parking_spot_id=ticket.parking_spot.spot_id,
            status="SUCCESS",
        )

        ticket.parking_spot.remove_vehicle()
        del self.active_tickets[ticket_id]

        return receipt


# ======================================================
# ENTRY AND EXIT GATES
# ======================================================

class EntryGate:
    def __init__(self, gate_id: str, parking_lot: ParkingLot):
        self.gate_id = gate_id
        self.parking_lot = parking_lot

    def scan_vehicle(self, vehicle: Vehicle) -> ParkingTicket:
        return self.parking_lot.park_vehicle(vehicle)


class ExitGate:
    def __init__(self, gate_id: str, parking_lot: ParkingLot):
        self.gate_id = gate_id
        self.parking_lot = parking_lot

    def process_exit(
        self,
        ticket_id: str,
        payment_method: str,
        pricing_context: PricingContext,
    ) -> Receipt:
        return self.parking_lot.exit_vehicle(
            ticket_id=ticket_id,
            payment_method=payment_method,
            pricing_context=pricing_context,
        )


## Design Summary

### Main flow

`EntryGate → ParkingLot → CompositeSpotAssignmentStrategy → ParkingPolicy → ParkingSpot → ParkingTicket`

`ExitGate → ParkingLot → PricingStrategyResolver → HourlyPricingStrategy → RateRepository → PaymentStrategyFactory → PaymentStrategy → PaymentResult → Receipt`

### Patterns used

- **Strategy Pattern**: PricingStrategy, PaymentStrategy, SpotAssignStrategy
- **Factory Pattern**: PaymentStrategyFactory using registry/decorator
- **Repository Pattern**: RateRepository
- **Policy Object**: ParkingPolicy
- **Dependency Injection**: ParkingLot receives resolver, factory, and assignment strategy


## Final End-to-End Tests

In [ ]:

# ======================================================
# FINAL END-TO-END TESTS
# ======================================================

def print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def print_ticket(ticket):
    print("Ticket ID:", ticket.ticket_id)
    print("Vehicle:", ticket.vehicle.license_plate)
    print("Vehicle Type:", ticket.vehicle.vehicle_type)
    print("Fuel Type:", ticket.vehicle.fuel_type)
    print("Spot ID:", ticket.parking_spot.spot_id)
    print("Spot Type:", ticket.parking_spot.spot_type)
    print("Spot Size:", ticket.parking_spot.spot_size)
    print("Has Charger:", ticket.parking_spot.has_charger)
    print("Ticket Status:", ticket.status)


def build_parking_lot():
    # Rate + pricing
    rate_repository = InMemoryRateRepository()
    hourly_pricing_strategy = HourlyPricingStrategy(rate_repository)

    pricing_strategy_resolver = PricingStrategyResolver()
    pricing_strategy_resolver.set_default_strategy(hourly_pricing_strategy)

    # Parking policy + assignment strategy
    parking_policy = ParkingPolicy()
    spot_assignment_strategy = CompositeSpotAssignmentStrategy(parking_policy)

    # Parking lot
    parking_lot = ParkingLot(
        parking_lot_id="P1",
        name="Mall Parking",
        address="Bangalore",
        pricing_strategy_resolver=pricing_strategy_resolver,
        payment_strategy_factory=PaymentStrategyFactory,
        spot_assignment_strategy=spot_assignment_strategy,
    )

    # Floors
    floor1 = ParkingFloor(floor_number=1)
    floor2 = ParkingFloor(floor_number=2)

    # Floor 1 spots
    floor1.add_parking_spot(
        ParkingSpot("F1-B1", VehicleSize.SMALL, SpotType.BIKE_SPOT, has_charger=False)
    )
    floor1.add_parking_spot(
        ParkingSpot("F1-B2", VehicleSize.SMALL, SpotType.BIKE_SPOT, has_charger=True)
    )
    floor1.add_parking_spot(
        ParkingSpot("F1-C1", VehicleSize.MEDIUM, SpotType.COMPACT_SPOT, has_charger=False)
    )
    floor1.add_parking_spot(
        ParkingSpot("F1-C2", VehicleSize.MEDIUM, SpotType.COMPACT_SPOT, has_charger=True)
    )
    floor1.add_parking_spot(
        ParkingSpot("F1-L1", VehicleSize.LARGE, SpotType.LARGE_SPOT, has_charger=False)
    )

    # Floor 2 spots
    floor2.add_parking_spot(
        ParkingSpot("F2-C1", VehicleSize.MEDIUM, SpotType.COMPACT_SPOT, has_charger=False)
    )
    floor2.add_parking_spot(
        ParkingSpot("F2-L1", VehicleSize.LARGE, SpotType.LARGE_SPOT, has_charger=False)
    )

    parking_lot.add_parking_floor(floor1)
    parking_lot.add_parking_floor(floor2)

    return parking_lot


print_section("SETUP")
parking_lot = build_parking_lot()
entry_gate = EntryGate("ENTRY-1", parking_lot)
exit_gate = ExitGate("EXIT-1", parking_lot)
print("Parking lot ready")


print_section("TEST 1: NORMAL CAR ENTRY")
car_ticket = entry_gate.scan_vehicle(Car("KA01AB1234"))
print_ticket(car_ticket)

assert car_ticket.parking_spot.spot_id == "F1-C1"
assert car_ticket.parking_spot.is_occupied is True
assert car_ticket.ticket_id in parking_lot.active_tickets
print("PASSED: Normal car got compact non-charging spot")


print_section("TEST 2: DUPLICATE VEHICLE ENTRY")
try:
    entry_gate.scan_vehicle(Car("KA01AB1234"))
    raise AssertionError("Duplicate vehicle entry should not be allowed")
except Exception as e:
    print("PASSED: Duplicate vehicle rejected")
    print("Reason:", e)


print_section("TEST 3: ELECTRIC CAR GETS CHARGING COMPACT SPOT")
electric_car_ticket = entry_gate.scan_vehicle(Car("KA02EC9999", FuelType.ELECTRIC))
print_ticket(electric_car_ticket)

assert electric_car_ticket.parking_spot.spot_id == "F1-C2"
assert electric_car_ticket.parking_spot.has_charger is True
assert electric_car_ticket.parking_spot.spot_type == SpotType.COMPACT_SPOT
print("PASSED: Electric car got compact charging spot")


print_section("TEST 4: ELECTRIC BIKE GETS BIKE CHARGING SPOT")
electric_bike_ticket = entry_gate.scan_vehicle(Motorcycle("KA03EB1111", FuelType.ELECTRIC))
print_ticket(electric_bike_ticket)

assert electric_bike_ticket.parking_spot.spot_id == "F1-B2"
assert electric_bike_ticket.parking_spot.has_charger is True
assert electric_bike_ticket.parking_spot.spot_type == SpotType.BIKE_SPOT
print("PASSED: Electric bike got bike charging spot")


print_section("TEST 5: TRUCK GETS LARGE SPOT")
truck_ticket = entry_gate.scan_vehicle(Truck("KA04TR7777"))
print_ticket(truck_ticket)

assert truck_ticket.parking_spot.spot_id == "F1-L1"
assert truck_ticket.parking_spot.spot_type == SpotType.LARGE_SPOT
print("PASSED: Truck got large spot")


print_section("TEST 6: NORMAL BIKE GETS NORMAL BIKE SPOT")
bike_ticket = entry_gate.scan_vehicle(Motorcycle("KA05BI2222"))
print_ticket(bike_ticket)

assert bike_ticket.parking_spot.spot_id == "F1-B1"
assert bike_ticket.parking_spot.spot_type == SpotType.BIKE_SPOT
print("PASSED: Normal bike got normal bike spot")


print_section("TEST 7: INVALID TICKET EXIT")
try:
    exit_gate.process_exit("INVALID-TICKET", "UPI", PricingContext())
    raise AssertionError("Invalid ticket exit should not be allowed")
except Exception as e:
    print("PASSED: Invalid ticket rejected")
    print("Reason:", e)


print_section("TEST 8: UNSUPPORTED PAYMENT METHOD")
try:
    exit_gate.process_exit(car_ticket.ticket_id, "BITCOIN", PricingContext())
    raise AssertionError("Unsupported payment method should not be allowed")
except Exception as e:
    print("PASSED: Unsupported payment method rejected")
    print("Reason:", e)

assert car_ticket.ticket_id in parking_lot.active_tickets
assert car_ticket.parking_spot.is_occupied is True
assert car_ticket.exit_time is None
print("PASSED: Ticket and spot stayed active after unsupported payment")


print_section("TEST 9: CAR EXIT SUCCESS")
car_ticket.entry_time = datetime.now() - timedelta(hours=2, minutes=10)

car_receipt = exit_gate.process_exit(
    ticket_id=car_ticket.ticket_id,
    payment_method="UPI",
    pricing_context=PricingContext(),
)

print(car_receipt)

assert car_receipt.amount_paid == 150  # Petrol car = 50/hour, 2h10m => 3 hours
assert car_ticket.status == TicketStatus.PAID
assert car_ticket.parking_spot.is_available() is True
assert car_ticket.ticket_id not in parking_lot.active_tickets
print("PASSED: Car exited successfully with correct receipt")


print_section("TEST 10: SAME TICKET EXIT TWICE")
try:
    exit_gate.process_exit(car_ticket.ticket_id, "UPI", PricingContext())
    raise AssertionError("Same ticket should not exit twice")
except Exception as e:
    print("PASSED: Same ticket exit rejected")
    print("Reason:", e)


print_section("TEST 11: EXIT ALL REMAINING VEHICLES")
remaining_tickets = [
    electric_car_ticket,
    electric_bike_ticket,
    truck_ticket,
    bike_ticket,
]

for ticket in remaining_tickets:
    ticket.entry_time = datetime.now() - timedelta(hours=1, minutes=5)

    receipt = exit_gate.process_exit(
        ticket_id=ticket.ticket_id,
        payment_method="CASH",
        pricing_context=PricingContext(),
    )

    print("\nExited:", ticket.vehicle.license_plate)
    print("Amount:", receipt.amount_paid)
    print("Payment:", receipt.payment_method)
    print("Transaction:", receipt.transaction_id)
    print("Spot free:", ticket.parking_spot.is_available())

    assert ticket.status == TicketStatus.PAID
    assert ticket.parking_spot.is_available() is True


print_section("TEST 12: FINAL SYSTEM STATE")
print("Active tickets:", parking_lot.active_tickets.keys())
assert len(parking_lot.active_tickets) == 0

all_spots = []
for floor in parking_lot.parking_floors:
    all_spots.extend(floor.parking_spots)

for spot in all_spots:
    print(
        spot.spot_id,
        "| occupied:",
        spot.is_occupied,
        "| charger:",
        spot.has_charger,
    )
    assert spot.is_occupied is False

print("\nALL TESTS PASSED SUCCESSFULLY")
